In [ ]:
# house_price_regression.ipynb

# Step 1: Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pickle
import os

# Step 2: Load Data
df = pd.read_csv("data/house_prices.csv")
print(df.head())

# Step 3: Check for missing values
print(df.isnull().sum())

# Step 4: Visualize data
sns.pairplot(df)
plt.show()

# Step 5: Handle Outliers (basic cap)
df = df[(df['Price'] < 1000000) & (df['Size'] < 5000)]

# Step 6: Feature & Target
X = df[['Size', 'Location', 'Rooms']]
y = df['Price']

# Step 7: Preprocessing Pipeline
preprocessor = ColumnTransformer([
    ('scale', MinMaxScaler(), ['Size', 'Rooms']),
    ('encode', OneHotEncoder(), ['Location'])
])

# Step 8: Model Pipeline
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('regressor', LinearRegression())
])

# Step 9: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 10: Train Model
pipeline.fit(X_train, y_train)

# Step 11: Predictions
y_pred = pipeline.predict(X_test)

# Step 12: Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("RMSE:", rmse)
print("R² Score:", r2)

# Step 13: Save Model
os.makedirs("model", exist_ok=True)
with open("model/linear_model.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# Step 14: Save Predictions
os.makedirs("output", exist_ok=True)
pred_df = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})
pred_df.to_csv("output/predictions.csv", index=False)
